# Chapter 2: Pre-Model Workflow and Data Preprocessing

## 📋 Summary

This chapter covers data preprocessing — arguably the most critical step in any ML pipeline. The guiding principle is **garbage in, garbage out**: no matter how powerful your model, poor quality data produces poor predictions. This chapter teaches you to handle the most common data quality issues using scikit-learn's preprocessing toolkit.

Key topics include handling **missing data** with `SimpleImputer`, `KNNImputer`, and `IterativeImputer`; **scaling features** with `StandardScaler`, `MinMaxScaler`, and `Normalizer`; **encoding categorical variables** with `OneHotEncoder` and `OrdinalEncoder`; building **preprocessing pipelines** with `Pipeline` and `ColumnTransformer`; and **feature engineering** to create more informative inputs for models.

The chapter emphasizes that preprocessing is not a one-time task but an ongoing responsibility, especially in production systems where new data arrives continuously.

---

## 🧠 Theoretical Explanation

### Missing Data
Missing data (NaN values) can occur from measurement errors, system failures, or deliberate omissions. Most scikit-learn algorithms cannot handle missing values directly. Three imputation strategies:
- **SimpleImputer**: Replaces with mean, median, or most frequent value. Fast but ignores relationships between features.
- **KNNImputer**: Uses K nearest neighbors to estimate missing values. Captures local structure but is computationally heavier.
- **IterativeImputer**: Models each feature with missing values as a function of all other features (multivariate regression). Most accurate but slowest.

### Feature Scaling
Many algorithms (KNN, SVM, gradient descent-based) are sensitive to feature scale. Two main approaches:
- **Standardization** (z-score): `z = (x - μ) / σ`. Centers around 0 with std=1. Good when data follows Gaussian distribution.
- **Normalization** (min-max): `x' = (x - min) / (max - min)`. Scales to [0,1]. Sensitive to outliers.
- **RobustScaler**: Uses median and IQR instead of mean/std. Robust to outliers.

### Categorical Encoding
ML algorithms require numerical inputs. Encoding methods:
- **OneHotEncoder**: Creates binary columns for each category. Avoids ordinal assumptions. Best for nominal data.
- **OrdinalEncoder**: Assigns integer codes. Only use when there is a natural order.
- **LabelEncoder**: Only for target variables (y), not features.

### ColumnTransformer
Real datasets have mixed types. `ColumnTransformer` applies different transformers to different columns, making it easy to preprocess numerical and categorical features separately within a single pipeline.


## 2.1 Handling Missing Data

In [ ]:
import numpy as np
import pandas as pd

# Create sample dataset with missing values
np.random.seed(2024)
n_samples = 20
n_features = 10

data = {f'Feature{i+1}': np.random.uniform(0, 100, n_samples) for i in range(n_features)}
df = pd.DataFrame(data)

# Introduce ~20% missing values
for column in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

print(f'Missing values per column:\n{df.isnull().sum()}')
df.head(6)

In [ ]:
from sklearn.impute import SimpleImputer

# Mean imputation
imputer = SimpleImputer(strategy='mean')
imputed_data = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)
print('SimpleImputer (mean) - no more NaN:', imputed_df.isnull().sum().sum())
imputed_df.head()

In [ ]:
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=2)
knn_imputed_data = knn_imputer.fit_transform(df)
knn_imputed_df = pd.DataFrame(knn_imputed_data, columns=df.columns)
print('KNNImputer - no more NaN:', knn_imputed_df.isnull().sum().sum())
knn_imputed_df.head()

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

iterative_imputer = IterativeImputer(random_state=42)
iterative_imputed_data = iterative_imputer.fit_transform(df)
iterative_imputed_df = pd.DataFrame(iterative_imputed_data, columns=df.columns)
print('IterativeImputer - no more NaN:', iterative_imputed_df.isnull().sum().sum())
iterative_imputed_df.head()

## 2.2 Scaling Techniques

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer, RobustScaler
import pandas as pd

# Use imputed data for scaling
data_to_scale = iterative_imputed_df.copy()

# StandardScaler: z = (x - mean) / std
std_scaler = StandardScaler()
std_scaled = pd.DataFrame(std_scaler.fit_transform(data_to_scale), columns=data_to_scale.columns)
print('StandardScaler - mean ~0:', std_scaled.mean().round(4).values[:3])
print('StandardScaler - std ~1:', std_scaled.std().round(4).values[:3])

In [ ]:
# MinMaxScaler: scales to [0, 1]
minmax_scaler = MinMaxScaler()
minmax_scaled = pd.DataFrame(minmax_scaler.fit_transform(data_to_scale), columns=data_to_scale.columns)
print('MinMaxScaler - min:', minmax_scaled.min().values[:3])
print('MinMaxScaler - max:', minmax_scaled.max().values[:3])

# Normalizer: normalizes each *sample* (row) to unit norm
normalizer = Normalizer()
norm_scaled = pd.DataFrame(normalizer.fit_transform(data_to_scale), columns=data_to_scale.columns)
print('\nNormalizer applied (row-wise normalization)')
norm_scaled.head(3)

## 2.3 Encoding Categorical Variables

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
import pandas as pd

# Sample categorical data
cat_data = pd.DataFrame({
    'color': ['red', 'blue', 'green', 'red', 'blue'],
    'size': ['small', 'large', 'medium', 'large', 'small']
})

# OneHotEncoder - creates binary columns
ohe = OneHotEncoder(sparse_output=False)
encoded = ohe.fit_transform(cat_data)
ohe_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
print('OneHotEncoded:\n', ohe_df)

In [ ]:
# OrdinalEncoder - assigns integer codes (use when order matters)
ord_enc = OrdinalEncoder(categories=[['small', 'medium', 'large']])
size_encoded = ord_enc.fit_transform(cat_data[['size']])
print('OrdinalEncoded size:', size_encoded.flatten())

## 2.4 Pipelines with ColumnTransformer

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

# Create a mixed-type dataset
np.random.seed(42)
n = 100
X = pd.DataFrame({
    'age': np.random.randint(20, 60, n).astype(float),
    'income': np.random.uniform(30000, 100000, n),
    'gender': np.random.choice(['male', 'female'], n),
    'city': np.random.choice(['NYC', 'LA', 'Chicago'], n)
})
y = np.random.randint(0, 2, n)

# Add some missing values
X.loc[np.random.choice(n, 10), 'age'] = np.nan

numerical_features = ['age', 'income']
categorical_features = ['gender', 'city']

# Define transformers for each type
numerical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numerical_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

# Full pipeline with model
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=200))
])

from sklearn.model_selection import cross_val_score
scores = cross_val_score(full_pipeline, X, y, cv=5, scoring='accuracy')
print(f'Cross-val accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})')

## 2.5 Feature Engineering

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

# Polynomial feature expansion
X = np.array([[1, 2], [3, 4], [5, 6]])
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)
print('Original shape:', X.shape)
print('After polynomial expansion:', X_poly.shape)
print('Feature names:', poly.get_feature_names_out(['x1', 'x2']))

## 🔑 Key Takeaways

- **Always handle missing data** before modeling — use `SimpleImputer` for speed, `KNNImputer` or `IterativeImputer` for accuracy.
- **Scale features** for distance-based or gradient-based algorithms. Use `StandardScaler` for normally distributed data, `MinMaxScaler` for bounded ranges, `RobustScaler` when outliers are present.
- **Encode categorical variables** with `OneHotEncoder` (nominal) or `OrdinalEncoder` (ordered).
- **Use `ColumnTransformer`** to handle mixed-type datasets cleanly.
- **Always fit preprocessors on training data only** and transform test data using the fitted objects to prevent data leakage.
- Wrap everything in a `Pipeline` for reproducibility and clean code.
